In [1]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /Users/gremy/Code/TSNN-1
tsnn module path: /Users/gremy/Code/TSNN-1/tsnn


In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn import tstorch
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
from tsnn.tstorch import transformers
import torch.nn.functional as F
import math
from typing import Optional

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

from torch import nn
from tqdm import tqdm
device = 'mps'


plt.style.use('ggplot')

In [3]:
# This is the new version of the notebook to make all the figures for the paper.

In [4]:
from tsnn.tstorch import models
from tsnn.benchmarks import torch_benchmarks
from tsnn.benchmarks.torch_benchmarks import MSELossWithL1Sparsity

# Defining the data parameters

In [5]:
# The first thing that we want to do is define all of parameters of the data we will be testing on.

For now we will be working with the following effects:
- Simple linear dependency
- TS shift with lags chosen randomly for each feature, but constant accros stocks
- CS shift, with lags chosen randomly for each stock and each feature
- TS-CS shift: combination of the two lags above
- Conditioning of one feature by another

We will fix the following global correlation levels: $\rho$ = 1,2,5,10,20,50. The first and last might not be too meaningful.

Next let's fix the size of the data that we want to test. We will always work with $T=4000$ points, and use $2500$ for training, $1500$ for testing. For the other dimensions:
- Default 1: n_ts = 10, n_rolling = 10, n_f = 20. This gives $\gamma = 0.8$.
- Default 2: n_ts = 10, n_rolling = 10, n_f = 5. This gives $\gamma = 0.2$.

Could also add two extra cases to test larger n_rolling or n_ts:
- Long TS: n_ts = 10, n_rolling = 40, n_f = 5. This gives $\gamma = 0.8$.
- Long CS: n_ts = 40, n_rolling = 10, n_f = 5. This gives $\gamma = 0.8$.

Next we need to compute a $\rho$ per feature. In the case where n_f=5, we will simply take $\frac{\rho}{\sqrt{5}}$ per feature. In the case n_f=20, we will take $\frac{\rho}{\sqrt{10}}$ for half of the features and $0$ for the half.

# Running the TC transformer

In [6]:
# Task one: let's run the best version of the TS-CS transformer on all of the above.

In [7]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def causal_mask_radius(b, h, q_idx, kv_idx):
    return (q_idx >= kv_idx) & (q_idx <= kv_idx+1)

def plot_mask(mask_fn, seq_len=20, title=None, device="cpu"):
    """
    Plot a binary attention mask defined by mask_fn(b,h,q_idx,kv_idx)
    as a (seq_len x seq_len) matrix.
    """
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device) 
    h = torch.zeros(1, device=device)

    mask = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])
    mask = mask.float().cpu()

    return pd.DataFrame(mask).style.background_gradient(axis=None).format(precision=0)

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool


In [8]:
def get_ols_corr(N_fea, T_train, rho, oos=True):
    ratio = N_fea/T_train
    if oos:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio/(1-ratio))
    else:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio)


In [9]:
def run_models(rhos, effect, T=4000, n_ts=10, n_f=20, n_rolling=10):

    embeddings_dict = {'d_lin': "none", 'd_cond': "none", 'd_shift': "T", 'd_cs': "C", 'd_cs_shift': "both", 'd_all': "both"}

    mask = causal_mask
    mask_c = build_attention_mask(mask, n_rolling, device=device)

    if n_f < 10:
        pct_zero_corr = 0
    else:
        pct_zero_corr = 0.5

    cs = [rho/np.sqrt((1-pct_zero_corr)*n_f) for rho in rhos]

    arg_list = {'d_lin': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
                'd_cond': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 1.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0},  
                'd_shift': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 1.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
                'd_cs': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 1.0, 'split_cs_shift': 0.0}, 
                'd_cs_shift': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 1.0}, 
                'd_all': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.2, 'split_shift': 0.2, 'split_seasonal': 0.0, 'split_cs': 0.2, 'split_cs_shift': 0.2}
               }
    
    z = generators.Generator(T, n_ts, n_f)
    
    res_train = []
    res_test = []
    for (i,c) in enumerate(cs):
        print("Running correl level:", rhos[i])
        theo_correl_is = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rhos[i], oos=False)
        theo_correl_oos = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rhos[i], oos=True)

        z.generate_dataset(
                pct_zero_corr=arg_list[effect]['pct_zero_corr'],
                split_conditional=arg_list[effect]['split_conditional'],
                split_shift=arg_list[effect]['split_shift'],
                split_seasonal=arg_list[effect]['split_seasonal'],
                split_cs=arg_list[effect]['split_cs'],
                split_cs_shift=arg_list[effect]['split_cs_shift'],
                random_ts_shift=n_rolling-1,
                low_corr=c,
                high_corr=c,
                shuffle_cs=True
            )

        z.get_dataloader()


        models_torch = {
            'full_attention_TC': models.CustomBiDimensionalTransformer(
                n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
                dropout=0.1, dim_feedforward=256, sparsify=None, roll_y=True, embeddings=embeddings_dict[effect],
            ).to(device),
            'full_attention_TCTC': models.CustomBiDimensionalTransformer(
                n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
                dropout=0.1, dim_feedforward=256, sparsify=None, roll_y=True, embeddings=embeddings_dict[effect],
            ).to(device),
        }

        models_torch = {
            k: torch_benchmarks.TorchWrapper(
                models_torch[k], 
                optimizer=torch.optim.AdamW(models_torch[k].parameters(), lr=0.001),
                loss_fn=nn.MSELoss()  
            ) for k in models_torch
        }


        z.get_dataloader(n_rolling=n_rolling, roll_y=True)
        for k in models_torch:
            models_torch[k].fit(z.train, test=z.test, epochs=40, plot=False, verbose=0)
            
      
        comp = benchmark_comparison.Comparator(models=[models_torch[k] for k in models_torch], 
        model_names=[k for k in models_torch]
        )

        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        for k in models_torch:

            train_corr = corr_train.loc[k, "optimal"]
            test_corr  = corr_test.loc[k, "optimal"]

            res_train.append({
                "rho": rhos[i],
                "model": k,
                "train_corr_optimal": train_corr
            })
            res_test.append({
                "rho": rhos[i],
                "model": k,
                "test_corr_optimal": test_corr
            })


        res_train.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "train_corr_optimal": theo_correl_is
            })
        res_test.append({
                "rho": rhos[i],
                "model": "theo_correl",
                "test_corr_optimal": theo_correl_oos
            })

    
    res_train = pd.DataFrame(res_train)
    res_test  = pd.DataFrame(res_test)

    res_train = res_train.pivot(index="rho", columns="model", values="train_corr_optimal")
    res_test  = res_test.pivot(index="rho", columns="model", values="test_corr_optimal")

    return res_train, res_test


In [57]:
tfull_shift_train, tfull_shift_test = run_models([0.02, 0.05, 0.1, 0.2, 0.5], 'd_shift')

Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [58]:
display(tfull_shift_train.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(tfull_shift_test.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.025,0.073,0.115,0.249,0.583
full_attention_TCTC,0.022,0.051,0.119,0.204,0.499
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.031,0.084,0.082,0.189,0.526
full_attention_TCTC,0.026,0.027,0.114,0.101,0.439
theo_correl,0.010,0.025,0.050,0.102,0.277


## Running on all effects, default data 1

In [10]:
tfull_lin_train, tfull_lin_test = run_models([0.02, 0.05, 0.1, 0.2, 0.5], 'd_lin')

Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [11]:
display(tfull_lin_train.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(tfull_lin_test.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.062,0.087,0.138,0.258,0.647
full_attention_TCTC,0.012,0.077,0.131,0.227,0.564
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.074,0.124,0.139,0.294,0.662
full_attention_TCTC,0.014,0.090,0.156,0.259,0.617
theo_correl,0.010,0.025,0.050,0.102,0.277


In [12]:
tfull_shift_train, tfull_shift_test = run_models([0.02, 0.05, 0.1, 0.2, 0.5], 'd_shift')

Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [13]:
display(tfull_shift_train.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(tfull_shift_test.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.012,0.049,0.111,0.250,0.540
full_attention_TCTC,0.015,0.050,0.106,0.231,0.525
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.007,0.033,0.079,0.197,0.440
full_attention_TCTC,0.008,0.022,0.066,0.200,0.397
theo_correl,0.010,0.025,0.050,0.102,0.277


In [12]:
tfull_cond_train, tfull_cond_test = run_models([0.02, 0.05, 0.1, 0.2, 0.5], 'd_cond')

Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [13]:
display(tfull_cond_train.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(tfull_cond_test.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.032,0.087,0.129,0.252,0.535
full_attention_TCTC,0.022,0.090,0.102,0.236,0.489
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.036,0.085,0.128,0.245,0.476
full_attention_TCTC,0.027,0.088,0.133,0.242,0.479
theo_correl,0.010,0.025,0.050,0.102,0.277


In [10]:
tfull_cs_train, tfull_cs_test = run_models([0.02, 0.05, 0.1, 0.2, 0.5], 'd_cs')

Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [11]:
display(tfull_cs_train.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(tfull_cs_test.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.028,0.056,0.115,0.225,0.519
full_attention_TCTC,0.020,0.048,0.106,0.207,0.471
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.019,0.053,0.088,0.204,0.510
full_attention_TCTC,0.019,0.062,0.082,0.224,0.530
theo_correl,0.010,0.025,0.050,0.102,0.277


In [13]:
tfull_cs_shift_train, tfull_cs_shift_test = run_models([0.02, 0.05, 0.1, 0.2, 0.5], 'd_cs_shift')

Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [14]:
display(tfull_cs_shift_train.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(tfull_cs_shift_test.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.020,0.043,0.088,0.156,0.359
full_attention_TCTC,0.023,0.041,0.094,0.186,0.394
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,-0.000,0.013,0.003,-0.004,0.001
full_attention_TCTC,0.002,-0.002,0.003,0.012,0.006
theo_correl,0.010,0.025,0.050,0.102,0.277


In [15]:
tfull_all_train, tfull_all_test = run_models([0.02, 0.05, 0.1, 0.2, 0.5], 'd_all')

Running correl level: 0.02


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.1


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.2


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running correl level: 0.5


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [17]:
display(tfull_all_train.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(tfull_all_test.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.038,0.081,0.124,0.231,0.518
full_attention_TCTC,0.029,0.073,0.115,0.214,0.476
theo_correl,0.022,0.056,0.112,0.222,0.542


rho,0.020000,0.050000,0.100000,0.200000,0.500000
model,,,,,
full_attention_TC,0.010,0.074,0.110,0.181,0.430
full_attention_TCTC,0.012,0.065,0.107,0.146,0.380
theo_correl,0.010,0.025,0.050,0.102,0.277


## Bootstrap

In [19]:

def run_bootstrap(rho, effect, n=5, T=4000, n_ts=10, n_f=20, n_rolling=10, models_torch=None):

    mask = causal_mask
    mask_c = build_attention_mask(mask, n_rolling, device=device)

    if n_f < 10:
        pct_zero_corr = 0
    else:
        pct_zero_corr = 0.5

    c = rho/np.sqrt((1-pct_zero_corr)*n_f)

    arg_list = {'d_lin': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
                'd_cond': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 1.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0},  
                'd_shift': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 1.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
                'd_cs': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 1.0, 'split_cs_shift': 0.0}, 
                'd_cs_shift': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 1.0}, 
                'd_all': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.2, 'split_shift': 0.2, 'split_seasonal': 0.0, 'split_cs': 0.2, 'split_cs_shift': 0.2}
               }
    
    z = generators.Generator(T, n_ts, n_f)
    
    res_train = []
    res_test = []
    for i in tqdm(range(n)):
        z.generate_dataset(
                pct_zero_corr=arg_list[effect]['pct_zero_corr'],
                split_conditional=arg_list[effect]['split_conditional'],
                split_shift=arg_list[effect]['split_shift'],
                split_seasonal=arg_list[effect]['split_seasonal'],
                split_cs=arg_list[effect]['split_cs'],
                split_cs_shift=arg_list[effect]['split_cs_shift'],
                random_ts_shift=n_rolling-1,
                low_corr=c,
                high_corr=c,
                shuffle_cs=True
            )

        z.get_dataloader()


        models_torch = {
            'full_attention_TC': models.CustomBiDimensionalTransformer(
                n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=8,
                dropout=0.1, dim_feedforward=256, sparsify=None, roll_y=True
            ).to(device),
            'full_attention_TCTC': models.CustomBiDimensionalTransformer(
                n_ts, n_f, n_rolling, mask=mask_c, layers='TCTC', nhead=8,
                dropout=0.1, dim_feedforward=256, sparsify=None, roll_y=True
            ).to(device),
        }

        models_torch = {
            k: torch_benchmarks.TorchWrapper(
                models_torch[k], 
                optimizer=torch.optim.AdamW(models_torch[k].parameters(), lr=0.001),
                loss_fn=nn.MSELoss()  
            ) for k in models_torch
        }


        z.get_dataloader(n_rolling=n_rolling, roll_y=True)
        for k in models_torch:
            models_torch[k].fit(z.train, test=z.test, epochs=10, plot=False, verbose=0)
            
      
        comp = benchmark_comparison.Comparator(models=[models_torch[k] for k in models_torch], 
        model_names=[k for k in models_torch]
        )
        
        # display(comp.correl(z, mode="test"))
        
        res_train.append(comp.correl(z, mode="train", return_values=True)['optimal'])
        res_test.append(comp.correl(z, mode="test", return_values=True)['optimal'])
        
    return res_train, res_test

In [25]:
#rhos = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
#effects = ['d_lin', 'd_shift', 'd_cs', 'd_cs_shift', 'd_cond', 'd_all']

rhos = [0.1]
effects = ['d_shift']

import warnings

for rho in rhos:
    for effect in effects:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            r_train, r_test = run_bootstrap(rho, effect, n=3)
        # pd.concat(r_train, axis=1).T.to_parquet(f'gr_results1/full_{effect}_{c}_train.parquet')
        # pd.concat(r_test, axis=1).T.to_parquet(f'gr_results1/full_{effect}_{c}_test.parquet')

100%|██████████| 3/3 [01:52<00:00, 37.36s/it]


In [26]:
r_train

[optimal                     NaN
 shift                  1.000000
 full_attention_TC      0.056089
 full_attention_TCTC    0.042788
 Name: optimal, dtype: float64,
 optimal                     NaN
 shift                  1.000000
 full_attention_TC      0.163623
 full_attention_TCTC    0.041542
 Name: optimal, dtype: float64,
 optimal                     NaN
 shift                  1.000000
 full_attention_TC      0.154661
 full_attention_TCTC    0.046472
 Name: optimal, dtype: float64]

In [28]:
r_test

[optimal                     NaN
 shift                  1.000000
 full_attention_TC      0.044705
 full_attention_TCTC    0.032436
 Name: optimal, dtype: float64,
 optimal                     NaN
 shift                  1.000000
 full_attention_TC      0.158884
 full_attention_TCTC    0.036102
 Name: optimal, dtype: float64,
 optimal                     NaN
 shift                  1.000000
 full_attention_TC      0.155003
 full_attention_TCTC    0.036472
 Name: optimal, dtype: float64]

# Test by hand

In [6]:
# Let's write some code here to test the models by hand. The first thing we will attempt is to get the sparsity to work on the CS shift model.

In [7]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def causal_mask_radius(b, h, q_idx, kv_idx):
    return (q_idx >= kv_idx) & (q_idx <= kv_idx+1)

def plot_mask(mask_fn, seq_len=20, title=None, device="cpu"):
    """
    Plot a binary attention mask defined by mask_fn(b,h,q_idx,kv_idx)
    as a (seq_len x seq_len) matrix.
    """
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device) 
    h = torch.zeros(1, device=device)

    mask = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])
    mask = mask.float().cpu()

    return pd.DataFrame(mask).style.background_gradient(axis=None).format(precision=0)

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool


In [8]:
def get_ols_corr(N_fea, T_train, rho, oos=True):
    ratio = N_fea/T_train
    if oos:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio/(1-ratio))
    else:
        return rho / np.sqrt(rho**2 + (1-rho**2) * ratio)


In [9]:
def keep_topk_per_row3(x, k=3):
    vals, idx = torch.topk(x, k=k, dim=-1, largest=True)
    out = torch.zeros_like(x)
    out.scatter_(-1, idx, 1)
    return out

def keep_topk_per_row5(x, k=3):
    vals, idx = torch.topk(x, k=k, dim=-1, largest=True)
    out = torch.zeros_like(x)
    out.scatter_(-1, idx, 1)
    return out

In [10]:
def keep_by_max_value2(x, frac=0.2):
    row_max = x.max(dim=-1, keepdim=True).values
    threshold = frac * row_max
    out = (x >= threshold).to(x.dtype)
    return out

def keep_by_max_value5(x, frac=0.2):
    row_max = x.max(dim=-1, keepdim=True).values
    threshold = frac * row_max
    out = (x >= threshold).to(x.dtype)
    return out


In [11]:
# Parameters

T=4000
n_rolling = 10
n_f = 5
n_ts = 40

rho = 0.05
effect = 'd_cs'


In [12]:
# Code to run the model with the parameters above.

embeddings_dict = {'d_lin': "none", 'd_cond': "none", 'd_shift': "T", 'd_cs': "C", 'd_cs_shift': "both", 'd_all': "both"}

mask = causal_mask
mask_c = build_attention_mask(mask, n_rolling, device=device)

if n_f < 10:
    pct_zero_corr = 0
else:
    pct_zero_corr = 0.5

c = rho/np.sqrt((1-pct_zero_corr)*n_f)

arg_list = {'d_lin': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
            'd_cond': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 1.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0},  
            'd_shift': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 1.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 0.0}, 
            'd_cs': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 1.0, 'split_cs_shift': 0.0}, 
            'd_cs_shift': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.0, 'split_shift': 0.0, 'split_seasonal': 0.0, 'split_cs': 0.0, 'split_cs_shift': 1.0}, 
            'd_all': {'pct_zero_corr': pct_zero_corr, 'split_conditional': 0.2, 'split_shift': 0.2, 'split_seasonal': 0.0, 'split_cs': 0.2, 'split_cs_shift': 0.2}
            }

z = generators.Generator(T, n_ts, n_f)

res_train = []
res_test = []

print("Running correl level:", rho)
theo_correl_is = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=False)
theo_correl_oos = get_ols_corr(n_ts*n_f*n_rolling, T*0.625, rho, oos=True)

z.generate_dataset(
        pct_zero_corr=arg_list[effect]['pct_zero_corr'],
        split_conditional=arg_list[effect]['split_conditional'],
        split_shift=arg_list[effect]['split_shift'],
        split_seasonal=arg_list[effect]['split_seasonal'],
        split_cs=arg_list[effect]['split_cs'],
        split_cs_shift=arg_list[effect]['split_cs_shift'],
        random_ts_shift=n_rolling-1,
        low_corr=c,
        high_corr=c,
        shuffle_cs=True
    )

z.get_dataloader()


models_torch = {
    'full_attention_CC': models.CustomBiDimensionalTransformer(
        n_ts, n_f, n_rolling, mask=mask_c, layers='CC', nhead=16,
        dropout=0.1, d_model=32, dim_feedforward=128, sparsify=None, roll_y=True, embeddings=embeddings_dict[effect],
    ).to(device),
    'top3_sparse_attention_CC': models.CustomBiDimensionalTransformer(
        n_ts, n_f, n_rolling, mask=mask_c, layers='CC', nhead=16,
        dropout=0.1, d_model=32, dim_feedforward=128, sparsify=keep_topk_per_row3, roll_y=True, embeddings=embeddings_dict[effect],
    ).to(device),
    'top5_sparse_attention_CC': models.CustomBiDimensionalTransformer(
        n_ts, n_f, n_rolling, mask=mask_c, layers='CC', nhead=16,
        dropout=0.1, d_model=32, dim_feedforward=128, sparsify=keep_topk_per_row5, roll_y=True, embeddings=embeddings_dict[effect],
    ).to(device),
    'max2_sparse_attention_CC': models.CustomBiDimensionalTransformer(
        n_ts, n_f, n_rolling, mask=mask_c, layers='CC', nhead=16,
        dropout=0.1, d_model=32, dim_feedforward=128, sparsify=keep_by_max_value2, roll_y=True, embeddings=embeddings_dict[effect],
    ).to(device),
    'max5_sparse_attention_CC': models.CustomBiDimensionalTransformer(
        n_ts, n_f, n_rolling, mask=mask_c, layers='CC', nhead=16,
        dropout=0.1, d_model=32, dim_feedforward=128, sparsify=keep_by_max_value5, roll_y=True, embeddings=embeddings_dict[effect],
    ).to(device),
    # 'top_sparse_attention_TC': models.CustomBiDimensionalTransformer(
    #     n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=1,
    #     dropout=0.1, dim_feedforward=256, sparsify=keep_topk_per_row, roll_y=True, embeddings=embeddings_dict[effect],
    # ).to(device),
    # 'max_sparse_attention_TC': models.CustomBiDimensionalTransformer(
    #     n_ts, n_f, n_rolling, mask=mask_c, layers='TC', nhead=1,
    #     dropout=0.1, dim_feedforward=256, sparsify=keep_by_max_value, roll_y=True, embeddings=embeddings_dict[effect],
    # ).to(device),

}

models_torch = {
    k: torch_benchmarks.TorchWrapper(
        models_torch[k], 
        optimizer=torch.optim.AdamW(models_torch[k].parameters(), lr=0.001),
        loss_fn=nn.MSELoss()  
    ) for k in models_torch
}


z.get_dataloader(n_rolling=n_rolling, roll_y=True)
for k in models_torch:
    models_torch[k].fit(z.train, test=z.test, epochs=40, plot=False, verbose=0)
    

comp = benchmark_comparison.Comparator(models=[models_torch[k] for k in models_torch], 
model_names=[k for k in models_torch]
)

corr_train = comp.correl(z, mode="train", return_values=True)
corr_test  = comp.correl(z, mode="test",  return_values=True)

for k in models_torch:

    train_corr = corr_train.loc[k, "optimal"]
    test_corr  = corr_test.loc[k, "optimal"]

    res_train.append({
        "rho": rho,
        "model": k,
        "train_corr_optimal": train_corr
    })
    res_test.append({
        "rho": rho,
        "model": k,
        "test_corr_optimal": test_corr
    })


res_train.append({
        "rho": rho,
        "model": "theo_correl",
        "train_corr_optimal": theo_correl_is
    })
res_test.append({
        "rho": rho,
        "model": "theo_correl",
        "test_corr_optimal": theo_correl_oos
    })


res_train = pd.DataFrame(res_train)
res_test  = pd.DataFrame(res_test)

res_train = res_train.pivot(index="rho", columns="model", values="train_corr_optimal")
res_test  = res_test.pivot(index="rho", columns="model", values="test_corr_optimal")

Running correl level: 0.05


/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [13]:
res_train

model,full_attention_CC,max2_sparse_attention_CC,max5_sparse_attention_CC,theo_correl,top3_sparse_attention_CC,top5_sparse_attention_CC
rho,,,,,,
0.05,0.015343,0.040966,0.014661,0.055884,0.043995,0.015274


In [14]:
res_test

model,full_attention_CC,max2_sparse_attention_CC,max5_sparse_attention_CC,theo_correl,top3_sparse_attention_CC,top5_sparse_attention_CC
rho,,,,,,
0.05,0.011246,0.041484,0.009541,0.025023,0.043989,0.008559


In [31]:
# What is the spasity doing here ? Let's try to visualize it..

In [43]:
pd.DataFrame(np.array(models_torch['top_sparse_attention_CC'].model.blocks[0].series.layers[0].self_attn.last_attn_weights.mean(0).mean(0).to('cpu'))).style.background_gradient(axis=0).format(precision=3)

/var/folders/vm/1x4t9m015pxd573nk50nfnkh0000gn/T/ipykernel_54236/748466161.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  pd.DataFrame(np.array(models_torch['top_sparse_attention_CC'].model.blocks[0].series.layers[0].self_attn.last_attn_weights.mean(0).mean(0).to('cpu'))).style.background_gradient(axis=0).format(precision=3)


,0,1,2,3,4,5,6,7,8,9
0,0.000,0.443,0.155,0.000,0.000,0.402,0.000,0.000,0.000,0.000
1,0.233,0.000,0.000,0.548,0.000,0.000,0.000,0.000,0.219,0.000
2,0.000,0.000,0.000,0.000,0.205,0.000,0.000,0.000,0.000,0.794
3,0.000,0.000,0.000,0.000,0.454,0.000,0.000,0.428,0.000,0.117
4,0.125,0.000,0.000,0.000,0.000,0.423,0.452,0.000,0.000,0.000
5,0.058,0.000,0.000,0.000,0.377,0.000,0.000,0.000,0.000,0.565
6,0.000,0.000,0.144,0.000,0.000,0.000,0.284,0.572,0.000,0.000
7,0.000,0.000,0.120,0.610,0.000,0.000,0.000,0.000,0.270,0.000
8,0.000,0.000,0.000,0.457,0.000,0.000,0.113,0.000,0.430,0.000
9,0.333,0.000,0.000,0.000,0.000,0.520,0.000,0.000,0.147,0.000


In [44]:
pd.DataFrame(np.array(models_torch['top_sparse_attention_CC'].model.blocks[1].series.layers[0].self_attn.last_attn_weights.mean(0).mean(0).to('cpu'))).style.background_gradient(axis=0).format(precision=3)

/var/folders/vm/1x4t9m015pxd573nk50nfnkh0000gn/T/ipykernel_54236/838903454.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  pd.DataFrame(np.array(models_torch['top_sparse_attention_CC'].model.blocks[1].series.layers[0].self_attn.last_attn_weights.mean(0).mean(0).to('cpu'))).style.background_gradient(axis=0).format(precision=3)


,0,1,2,3,4,5,6,7,8,9
0,0.229,0.577,0.000,0.000,0.000,0.194,0.000,0.000,0.000,0.000
1,0.000,0.000,0.000,0.000,0.000,0.420,0.238,0.342,0.000,0.000
2,0.000,0.000,0.000,0.067,0.000,0.000,0.558,0.000,0.374,0.000
3,0.150,0.000,0.000,0.722,0.000,0.000,0.127,0.000,0.000,0.000
4,0.000,0.489,0.000,0.000,0.000,0.000,0.266,0.244,0.000,0.000
5,0.000,0.588,0.000,0.000,0.000,0.261,0.000,0.151,0.000,0.000
6,0.000,0.369,0.000,0.393,0.238,0.000,0.000,0.000,0.000,0.000
7,0.000,0.000,0.000,0.000,0.183,0.000,0.428,0.000,0.389,0.000
8,0.000,0.000,0.000,0.201,0.190,0.609,0.000,0.000,0.000,0.000
9,0.000,0.000,0.000,0.000,0.118,0.000,0.215,0.667,0.000,0.000


In [46]:
pd.DataFrame(np.array(models_torch['max_sparse_attention_CC'].model.blocks[0].series.layers[0].self_attn.last_attn_weights.mean(0).mean(0).to('cpu'))).style.background_gradient(axis=0).format(precision=3)

/var/folders/vm/1x4t9m015pxd573nk50nfnkh0000gn/T/ipykernel_54236/3075308818.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  pd.DataFrame(np.array(models_torch['max_sparse_attention_CC'].model.blocks[0].series.layers[0].self_attn.last_attn_weights.mean(0).mean(0).to('cpu'))).style.background_gradient(axis=0).format(precision=3)


,0,1,2,3,4,5,6,7,8,9
0,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000
1,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,0.000,0.000,0.000,0.000,0.000,0.264,0.408,0.000,0.000,0.328
3,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
4,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000
5,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000
6,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
7,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000
8,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000
9,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


In [47]:
pd.DataFrame(np.array(models_torch['max_sparse_attention_CC'].model.blocks[1].series.layers[0].self_attn.last_attn_weights.mean(0).mean(0).to('cpu'))).style.background_gradient(axis=0).format(precision=3)

/var/folders/vm/1x4t9m015pxd573nk50nfnkh0000gn/T/ipykernel_54236/2746487211.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  pd.DataFrame(np.array(models_torch['max_sparse_attention_CC'].model.blocks[1].series.layers[0].self_attn.last_attn_weights.mean(0).mean(0).to('cpu'))).style.background_gradient(axis=0).format(precision=3)


,0,1,2,3,4,5,6,7,8,9
0,0.000,0.000,0.436,0.000,0.000,0.000,0.564,0.000,0.000,0.000
1,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000
2,0.323,0.000,0.000,0.000,0.000,0.000,0.271,0.406,0.000,0.000
3,0.354,0.000,0.000,0.000,0.434,0.000,0.000,0.212,0.000,0.000
4,0.000,0.000,0.190,0.323,0.487,0.000,0.000,0.000,0.000,0.000
5,0.000,0.000,0.382,0.000,0.000,0.000,0.000,0.000,0.000,0.618
6,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000
7,0.000,0.000,0.222,0.000,0.000,0.000,0.000,0.474,0.000,0.304
8,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000
9,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000
